# 04 — Joint curriculum on the same general model

Continue the exact same OneCore weights, optimizer and scheduler after the initial language/tool phase. This phase integrates image, audio, video, motion, text and tool/action records into a single joint training stream. It is not a standalone media model and does not create modality-specific checkpoints.


In [ ]:
import os, sys, json
from pathlib import Path


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        for project in (candidate, candidate / "onecore-ai"):
            if (project / "configs" / "phone-research.yaml").is_file() and (project / "src" / "onecore").is_dir():
                return project
    raise RuntimeError(f"Could not locate OneCore project root from {current}")


root = find_project_root()
print("OneCore project root:", root)
sys.path.insert(0, str(root / "src"))

from onecore.acquire import acquire_sources
from onecore.crawlers import notebook_cursor_summary, source_ids_for_notebook
from onecore.train import run_training

NOTEBOOK_ID = "04_multimodal_curriculum"
print("Owned streaming data cursors:", json.dumps(
    notebook_cursor_summary(root, NOTEBOOK_ID), ensure_ascii=False, indent=2
))

if os.environ.get("ONECORE_ACQUIRE_ONLY", "0") == "1":
    source_ids = source_ids_for_notebook(root, NOTEBOOK_ID)
    if not source_ids:
        raise RuntimeError(f"No enabled source is assigned to {NOTEBOOK_ID}")
    budget_seconds = int(os.environ.get("ONECORE_ACQUIRE_BUDGET_SECONDS", "0"))
    source_budget_seconds = int(os.environ.get("ONECORE_ACQUIRE_SOURCE_BUDGET_SECONDS", "150"))
    report = acquire_sources(
        root,
        profile=os.environ.get("ONECORE_DATA_PROFILE", "expanded"),
        only_sources=source_ids,
        time_budget_seconds=budget_seconds or None,
        source_budget_seconds=source_budget_seconds or None,
    )
    print(json.dumps({
        "notebook": NOTEBOOK_ID,
        "source_count": len(source_ids),
        "source_ids": source_ids,
        "acquisition_complete": report.get("acquisition_complete"),
        "pending_sources": report.get("pending_sources"),
        "total_new_records": report.get("total_new_records"),
        "cursors": notebook_cursor_summary(root, NOTEBOOK_ID),
    }, ensure_ascii=False, indent=2))
else:
    if os.environ.get("ONECORE_RESUME", "false").lower() != "true":
        raise RuntimeError("Joint curriculum must resume the exact same OneCore checkpoint; refusing a fresh separate model.")
    report = run_training(
        project_root=root,
        config_path=root / "configs" / "phone-research.yaml",
        stage="multimodal",
        steps=int(os.environ.get("ONECORE_STEPS", "10")),
        output_dir=root / "run-state",
        resume=True,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))
